# Adaptive ensemble trading study

Explore the earned forecasts and simulated fills from the 2023–2025 chronological evaluation. The final cell can train the full study from public prices and the recorded language-model evidence.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import Image, display

root = Path.cwd()
if not (root / "ceci2_demo" / "protocol.json").exists():
    if (root.parent / "ceci2_demo" / "protocol.json").exists():
        root = root.parent
    else:
        root = Path("/content/ML-TradeEngine")
        if not root.exists():
            subprocess.run(["git", "clone", "--depth=1", "https://github.com/Yash789k/ML-TradeEngine.git", str(root)], check=True)
study = root / "data/ceci2-demo/five-session"
display(Image(filename=str(study / "datasets.png")))
display(Image(filename=str(study / "sources.png")))

## What did each $10,000 portfolio earn?

The benchmarks use the same dates and fee schedule. SPY buys at the first eligible opening in each annual fold and sells at its end. Exposure-matched SPY holds the same fraction of capital in equities as the full gate.

In [ ]:
comparison = pd.read_csv(study / "comparison.csv").set_index("model")
display(comparison[["net_return", "max_drawdown", "sharpe", "mean_exposure", "log_loss"]].round(4))
display(Image(filename=str(study / "backtest.png")))

## Inspect a decision and its fills

In [ ]:
tape = pd.read_csv(study / "predictions.csv.gz")
last = tape.decision.max()
display(tape.query("model == 'HMM gate' and decision == @last")[["symbol","down","flat","up","label"]].head(10))
orders = pd.read_csv(study / "dynamic_orders.csv.gz")
display(orders.tail(12))

## Compare the agent's proposal with later outcomes

In [ ]:
import json
reviews = pd.DataFrame(json.loads((study / "adaptation.json").read_text()))
display(reviews[["year","method","proposed_l2","accepted","lower_90pct","trades"]])
display(pd.read_csv(study / "costs.csv").query("model == 'HMM gate'"))

## Retrain on public data

Set `retrain` to True to install the training dependencies in an isolated environment and run the five-session comparison. The default uses recorded agent decisions. A fresh research run takes an empty output directory.

In [ ]:
retrain = False
if retrain:
    import tempfile
    environment = Path(tempfile.mkdtemp(prefix="ceci2-training-"))
    subprocess.run([sys.executable, "-m", "venv", str(environment / "env")], check=True)
    python = str(environment / "env/bin/python")
    subprocess.run([python, "-m", "pip", "install", "-r", "ceci2_demo/requirements.txt"], cwd=root, check=True)
    output = environment / "study"
    subprocess.run([python, "-m", "ceci2_demo.run", "--device", "auto", "--output", str(output)], cwd=root, check=True)
    display(pd.read_csv(output / "comparison.csv"))